# Final model generalization on sex

## Imports

In [ ]:
#Install requirements
%pip install -r "../Requirements.txt"

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import numpy as np
import sys
import warnings
from sklearn.pipeline import Pipeline

project_dir = Path.cwd().parent

sys.path.insert(0, str(Path.cwd().parent)) # import src/
warnings.filterwarnings("ignore")

pd.set_option("display.width", 220, "display.max_columns", 30)

from src import (NestedCVRegressor, default_models,
                           MRMRSelector, save_model)

from src import best_params_from_result, build_xy, load_anndata, load_per_family, load_model

HVG_DATA_DIR = project_dir / 'data' / 'generalisability'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Datasets

In [3]:
FAMILIES_FEMALE = {
    "KenyonCells":  (HVG_DATA_DIR / "Kenyon_cells__sex_train_female_test_male_train.h5ad",  HVG_DATA_DIR / "Kenyon_cells__sex_train_female_test_male_test.h5ad"),
    "OpticLobe":    (HVG_DATA_DIR / "Optic_lobe__sex_train_female_test_male_train.h5ad",    HVG_DATA_DIR / "Optic_lobe__sex_train_female_test_male_test.h5ad"),
    "Monoaminergic":(HVG_DATA_DIR / "Monoaminergic__sex_train_female_test_male_train.h5ad", HVG_DATA_DIR / "Monoaminergic__sex_train_female_test_male_test.h5ad"),
    "Glia":         (HVG_DATA_DIR / "Glia__sex_train_female_test_male_train.h5ad",          HVG_DATA_DIR / "Glia__sex_train_female_test_male_test.h5ad"),
    "Peptidergic":  (HVG_DATA_DIR / "Peptidergic__sex_train_female_test_male_train.h5ad",   HVG_DATA_DIR / "Peptidergic__sex_train_female_test_male_test.h5ad"),
    "Clock":        (HVG_DATA_DIR / "Clock__sex_train_female_test_male_train.h5ad",         HVG_DATA_DIR / "Clock__sex_train_female_test_male_test.h5ad"),
}
FAMILIES_FEMALE = {f: p for f, p in FAMILIES_FEMALE.items() if p[0].exists() and p[1].exists()}

FAMILIES_MALE = {
    "KenyonCells":  (HVG_DATA_DIR / "Kenyon_cells__sex_train_male_test_female_train.h5ad",  HVG_DATA_DIR / "Kenyon_cells__sex_train_male_test_female_test.h5ad"),
    "OpticLobe":    (HVG_DATA_DIR / "Optic_lobe__sex_train_male_test_female_train.h5ad",    HVG_DATA_DIR / "Optic_lobe__sex_train_male_test_female_test.h5ad"),
    "Monoaminergic":(HVG_DATA_DIR / "Monoaminergic__sex_train_male_test_female_train.h5ad", HVG_DATA_DIR / "Monoaminergic__sex_train_male_test_female_test.h5ad"),
    "Glia":         (HVG_DATA_DIR / "Glia__sex_train_male_test_female_train.h5ad",          HVG_DATA_DIR / "Glia__sex_train_male_test_female_test.h5ad"),
    "Peptidergic":  (HVG_DATA_DIR / "Peptidergic__sex_train_male_test_female_train.h5ad",   HVG_DATA_DIR / "Peptidergic__sex_train_male_test_female_test.h5ad"),
    "Clock":        (HVG_DATA_DIR / "Clock__sex_train_male_test_female_train.h5ad",         HVG_DATA_DIR / "Clock__sex_train_male_test_female_test.h5ad"),
}
FAMILIES_MALE = {f: p for f, p in FAMILIES_MALE.items() if p[0].exists() and p[1].exists()}

## Final model fit and save with fs and best hyperparameters

In [4]:
final_model_per_family = load_per_family("../results/final_model_per_family",
                                         name="final_model_per_family")

In [5]:

# Train on female
for fam, (train, _test) in FAMILIES_FEMALE.items():
    X, y, _ = build_xy(load_anndata(train), extra_obs=("nUMI",))
    bp = best_params_from_result(final_model_per_family.results[fam], "XGBoost", metric="RMSE")
    pipe_female = Pipeline([("selector", MRMRSelector(k=100, relevance="f")),
                            ("model", default_models()["XGBoost"].set_params(**bp))])
    pipe_female.fit(X, y) # mRMR picks 100 genes on ALL train, then XGB fits
    save_model(pipe_female, f"../results/final_model_per_family/{fam}/final_XGBoost_generalize_sex_female.joblib")

In [6]:

# Train on male
for fam, (train, _test) in FAMILIES_MALE.items():
    X, y, _ = build_xy(load_anndata(train), extra_obs=("nUMI",))
    bp = best_params_from_result(final_model_per_family.results[fam], "XGBoost", metric="RMSE")
    pipe_male = Pipeline([("selector", MRMRSelector(k=100, relevance="f")),
                          ("model", default_models()["XGBoost"].set_params(**bp))])
    pipe_male.fit(X, y) # mRMR picks 100 genes on ALL train, then XGB fits
    save_model(pipe_male, f"../results/final_model_per_family/{fam}/final_XGBoost_generalize_sex_male.joblib")

## Evaluate on test

In [7]:
# Predict on male test sets
test_scores_female_male = {}
for fam, (train, test) in FAMILIES_FEMALE.items():
    Xte, yte, _ = build_xy(load_anndata(test), extra_obs=("nUMI",))
    pipe_female_eval = load_model(f"../results/final_model_per_family/{fam}/final_XGBoost_generalize_sex_female.joblib")
    pred_male = pipe_female_eval.predict(Xte)
    test_scores_female_male[fam] = {"MAE": float(np.mean(np.abs(yte - pred_male))),
                                    "RMSE": float(np.sqrt(np.mean((yte - pred_male) ** 2)))}
display(test_scores_female_male)

{'KenyonCells': {'MAE': 2.561219431985026, 'RMSE': 4.43130576226299},
 'OpticLobe': {'MAE': 2.9062311506238516, 'RMSE': 5.210619635349539},
 'Monoaminergic': {'MAE': 4.951015254571324, 'RMSE': 8.120585512716614},
 'Glia': {'MAE': 5.832007131454251, 'RMSE': 8.371351141873971},
 'Peptidergic': {'MAE': 6.759010615863845, 'RMSE': 9.688280267408178},
 'Clock': {'MAE': 5.741288547490199, 'RMSE': 8.482263783566795}}

In [8]:
# Predict on female test sets
test_scores_male_female = {}
for fam, (train, test) in FAMILIES_MALE.items():
    Xte, yte, _ = build_xy(load_anndata(test), extra_obs=("nUMI",))
    pipe_male_eval = load_model(f"../results/final_model_per_family/{fam}/final_XGBoost_generalize_sex_male.joblib")
    pred_female = pipe_male_eval.predict(Xte)
    test_scores_male_female[fam] = {"MAE": float(np.mean(np.abs(yte - pred_female))),
                                    "RMSE": float(np.sqrt(np.mean((yte - pred_female) ** 2)))}
display(test_scores_male_female)

{'KenyonCells': {'MAE': 3.5580858496640704, 'RMSE': 6.48045734320531},
 'OpticLobe': {'MAE': 3.2656704622995876, 'RMSE': 5.317815991672395},
 'Monoaminergic': {'MAE': 5.41171647919507, 'RMSE': 8.580551120376107},
 'Glia': {'MAE': 5.945743098959989, 'RMSE': 8.221854904017217},
 'Peptidergic': {'MAE': 4.862913487730538, 'RMSE': 7.206473080010076},
 'Clock': {'MAE': 6.398112656687648, 'RMSE': 10.299757469795054}}